# Aula 5: evolução do IDH por estado

Atividade com pandas e matplotlib: leitura, limpeza, ranking de 2024, comparação com 1991, formato longo e gráficos.

**Como executar:** coloque `Tabela4.csv` na mesma pasta deste notebook (no Colab, envie também o CSV) e execute as células na ordem. Ajuste `path` se necessário.

**Situação dos dados:** o CSV não acompanhou o notebook recebido. Os resultados antigos eram apenas uma prévia e foram removidos para não parecerem resultados desta versão. Os cálculos e gráficos abaixo serão gerados ao executar com o arquivo completo.


In [ ]:
# No terminal, se necessário: python -m pip install pandas matplotlib
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

path = Path("Tabela4.csv")


## 1. Abrir o CSV

Conforme a configuração do notebook original, a primeira linha é um título e a segunda é o cabeçalho. `sep=";"` indica o separador; `decimal=","` reconhece a vírgula decimal; `encoding="latin-1"` permite ler os acentos do arquivo.


In [ ]:
if not path.is_file():
    raise FileNotFoundError("Coloque Tabela4.csv na pasta do notebook ou ajuste path.")

df = pd.read_csv(path, sep=";", skiprows=1, decimal=",", encoding="latin-1")


## 2. Limpar colunas vazias e preparar os valores

Removemos colunas e linhas totalmente vazias, inclusive as preenchidas apenas com espaços. Valores não numéricos nos anos ficam ausentes (`NaN`), nunca zero.


In [ ]:
df = df.replace(r"^\s*$", pd.NA, regex=True)
df = df.dropna(axis=1, how="all").dropna(axis=0, how="all")
df.columns = df.columns.astype(str).str.strip()

anos = sorted([c for c in df.columns if str(c).isdigit()], key=int)
for coluna in anos:
    df[coluna] = pd.to_numeric(
        df[coluna].astype(str).str.strip().str.replace(",", ".", regex=False),
        errors="coerce",
    )

obrigatorias = {"Sigla", "Estado", "1991", "2024"}
if not obrigatorias.issubset(df.columns):
    raise ValueError(f"Colunas ausentes: {obrigatorias - set(df.columns)}")
df["Sigla"] = df["Sigla"].astype("string").str.strip().str.upper()
df["Estado"] = df["Estado"].astype("string").str.strip()
display(df.head())
df.info()


## 3. Ordenar as unidades da federação pelo maior IDH em 2024

`ascending=False` ordena do maior para o menor. Valores ausentes ficam no final. A tabela inclui o Distrito Federal, caso esteja no CSV.


In [ ]:
ranking_2024 = df.sort_values("2024", ascending=False, na_position="last")
display(ranking_2024[["Sigla", "Estado", "2024"]].reset_index(drop=True))


## 4. Qual UF teve a maior melhora entre 1991 e 2024?

A melhora é a diferença absoluta: **IDH de 2024 − IDH de 1991**, em pontos do índice. Não é uma variação percentual. Todas as UFs empatadas no maior ganho são exibidas.


In [ ]:
comparacao = df[["Sigla", "Estado", "1991", "2024"]].copy()
comparacao["Melhora"] = (comparacao["2024"] - comparacao["1991"]).round(6)
validos = comparacao.dropna(subset=["1991", "2024"])
if validos.empty:
    print("Não há dados suficientes para comparar 1991 e 2024.")
else:
    maior_melhora = validos["Melhora"].max()
    if maior_melhora > 0:
        print("Maior melhora entre 1991 e 2024:")
        display(validos.loc[validos["Melhora"].eq(maior_melhora)])
    else:
        print("Nenhuma UF com dados completos apresentou melhora.")

display(comparacao.sort_values("Melhora", ascending=False).reset_index(drop=True))
if len(validos) < len(comparacao):
    print("Atenção: há UFs sem dados nos dois anos; a conclusão é parcial.")


## 5. Existe alguma UF em que o IDH piorou?

Primeiro, comparamos **2024 com 1991**. Uma diferença negativa indica piora. Uma melhora no período completo não impede quedas em anos intermediários; verificaremos isso após o `melt`.


In [ ]:
pioraram = validos.loc[validos["Melhora"] < 0]
if validos.empty:
    print("Não há dados suficientes para responder.")
elif pioraram.empty:
    print("Nenhuma UF com dados nos dois anos piorou de 1991 para 2024.")
else:
    print("UFs que pioraram de 1991 para 2024:")
    display(pioraram)


## 6. Transformar do formato largo para o longo com melt

No formato largo, cada ano é uma coluna. No longo, cada linha representa uma UF em um ano. `id_vars` mantém os identificadores; `value_vars` seleciona os anos; `var_name` e `value_name` dão nomes às novas colunas.


In [ ]:
anos = [c for c in df.columns if str(c).isdigit()]
print(anos)

id_vars = [c for c in df.columns if c not in anos]
df_longo = df.melt(
    id_vars=id_vars,
    value_vars=anos,
    var_name="Ano",
    value_name="IDH",
)
df_longo["Ano"] = df_longo["Ano"].astype(int)
df_longo["IDH"] = pd.to_numeric(df_longo["IDH"], errors="coerce")
df_longo.head()

### Verificar quedas entre observações consecutivas

`diff()` subtrai o IDH da observação anterior de cada UF. Como a série tem intervalos (por exemplo, 1991–2000), uma observação anterior nem sempre é o ano imediatamente anterior. Valores ausentes não são preenchidos.


In [ ]:
evolucao = df_longo.sort_values(["Sigla", "Ano"]).copy()
evolucao["Ano anterior"] = evolucao.groupby("Sigla")["Ano"].shift(1).astype("Int64")
evolucao["Variacao"] = evolucao.groupby("Sigla")["IDH"].diff().round(6)
quedas = evolucao.loc[evolucao["Variacao"] < 0,
    ["Sigla", "Estado", "Ano anterior", "Ano", "IDH", "Variacao"]]
if quedas.empty:
    print("Não foram identificadas quedas nas comparações disponíveis.")
else:
    print("UFs com queda em pelo menos um intervalo:")
    display(quedas[["Sigla", "Estado"]].drop_duplicates().reset_index(drop=True))
    display(quedas.reset_index(drop=True))


## 7. Plotar apenas Minas Gerais

Filtramos a sigla `MG` e ordenamos por ano antes de ligar os pontos.


In [ ]:
mg = df_longo.loc[df_longo["Sigla"].eq("MG")].sort_values("Ano")
if mg["IDH"].notna().sum() == 0:
    print("Não há dados de Minas Gerais para plotar.")
else:
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(mg["Ano"], mg["IDH"], marker="o", linewidth=2, label="Minas Gerais")
    ax.set_title("Evolução do IDH de Minas Gerais (1991–2024)")
    ax.set_xlabel("Ano")
    ax.set_ylabel("IDH")
    ax.set_ylim(0.3, 0.9)
    ax.grid(alpha=0.3)
    ax.legend()
    fig.tight_layout()
    plt.show()


## 8. Plotar a evolução do IDH de cada UF

`groupby("Sigla")` separa os dados por UF e o laço desenha uma linha para cada grupo.


In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))

for sigla, grupo in df_longo.groupby("Sigla"):
    grupo = grupo.sort_values("Ano")
    ax.plot(grupo["Ano"], grupo["IDH"], marker="o", markersize=3, linewidth=1.5, label=sigla)

ax.set_title("Evolução do IDH por estado (1991–2024)")
ax.set_xlabel("Ano")
ax.set_ylabel("IDH")
ax.set_ylim(0.3, 0.9)
ax.legend(ncol=3, bbox_to_anchor=(1.02, 1), loc="upper left", title="UF")
fig.tight_layout()
plt.show()